# FHRPY — CTG viewer & analysis demo

Analyse a full cardiotocography (CTG) recording, then compare the **expert
labels** with the **FHRPY methods** (false-signal detection and WMFB baseline),
all inline under **VSCode** and **Google Colab**.

> **Run the setup cell below first.** It makes `fhrpy` importable on its own —
> no manual `pip install` from a checkout of
> [data-coeur/FHRPY](https://github.com/data-coeur/FHRPY).

In [ ]:
# === Setup — run this cell first (makes the notebook self-contained) ===
# If you just pulled new code, RESTART THE KERNEL first: Python caches modules,
# so a previously-imported `fhrpy` (e.g. an older dataset list) stays in memory.
import sys, pathlib
# Prefer the local checkout (so the bundled datasets/examples are the current
# ones) by putting the repo root FIRST on sys.path.
_root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (p / "fhrpy" / "__init__.py").exists()), None)
if _root and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
# Self-heal a stale kernel: drop any fhrpy already imported (e.g. before a git
# pull) so re-running this cell uses the current code on disk — no restart needed.
for _m in [k for k in list(sys.modules) if k == "fhrpy" or k.startswith("fhrpy.")]:
    del sys.modules[_m]
try:
    import fhrpy
except ModuleNotFoundError:
    # Standalone (e.g. Google Colab): install the released package from GitHub.
    # The `main` branch is public, so NO token is required.
    %pip install -q "fhrpy[viewer] @ git+https://github.com/data-coeur/FHRPY.git@main"
    import fhrpy

import fhrpy.datasets as _ds
print("fhrpy", fhrpy.__version__, "from", fhrpy.__file__)
if not any(e["name"].startswith("ctg_") for e in _ds.list_examples()):
    print("\n⚠️  A stale `fhrpy` is cached — RESTART THE KERNEL and re-run this cell.")

## 1. Analyse a complete CTG

By default we open one of the FHRMA **Examples** recordings (a full Doppler CTG)
and run the **whole FHRPY pipeline**: false-signal detection & removal → WMFB
baseline → accelerations / decelerations. The blue **Expulsion** line is the
protected `£Expulsion` marker (its `£` prefix makes it non-editable), read from
the recording — no need to pass the expulsion time.

Change `EXAMPLE` to any `ctg_*` name listed above. Use the toolbar to toggle the
baseline / contraction / false-signal zones, switch 1↔3 cm/min, measure, or
print to PDF.

In [ ]:
import fhrpy.datasets as ds
from fhrpy.viewer import FHRViewer, link_scroll
from IPython.display import display, HTML

# The FULL FHRMA datasets are bundled (no manifest; name = handle). Inventory:
#   by category -> {'ctg': 11, 'morpho': 156, 'fs': 1065}   (1232 total, 694 labelled)
#   morpho_train01..66  (expert labels)   |  morpho_test01..90  (held-out)
#   fs_dopmhr_train.../val/testcp/testdbs  |  fs_scalp_train.../val/test
#   ctg_example_01..11  (the FHRMA Examples)
# Use ds.example_names("morpho"|"fs"|"ctg") to enumerate; the name IS the handle.

# Pick the CTG to analyse: one line uncommented (the default), the others alternatives.
EXAMPLE = "ctg_example_01"
# EXAMPLE = "ctg_example_02"
# EXAMPLE = "ctg_example_03"
# EXAMPLE = "ctg_example_04"
# EXAMPLE = "ctg_example_05"
# EXAMPLE = "ctg_example_06"
# EXAMPLE = "ctg_example_07"
# EXAMPLE = "ctg_example_08"
# EXAMPLE = "ctg_example_09"
# EXAMPLE = "ctg_example_10"
# EXAMPLE = "ctg_example_11"

ds.load_and_process(EXAMPLE, source="method", height=480)

## 2. Expert labels vs prediction — **false signals**

Here we use a **training** record (which carries the expert ground truth) and
show, side by side, the **expert** false-signal episodes and the **FHRPY**
detector. `link_scroll(...)` keeps both windows at the **same instant**: scroll,
page or wheel-scroll either one and the other follows — so label/prediction
differences line up.

> **False-signal detection requirements.** Doppler detection expects `FHR1` = the
> **Doppler** channel and `FHR2` = the **scalp** (direct fetal ECG) channel. The
> maternal-heart-rate channel (`MHR`) must be **time-aligned** to the FHR first:
> on Philips monitors the maternal pulse lags by about **12.5 s** with the SpO2
> **oximeter**, or about **5 s** when derived from the **TOCO/belt** — shift `MHR`
> by that delay before running detection.

In [ ]:
NAME = "fs_dopmhr_train0006"   # a DopMHR training record with expert FS labels
fs_label = ds.load_and_process(NAME, source="expert", channels=["FHR1", "MHR"], height=480)
# source="fs": false-signal detection ONLY (no baseline / contractions here).
fs_pred  = ds.load_and_process(NAME, source="fs", channels=["FHR1", "MHR"], height=480)

display(HTML("<b>Expert false-signal labels</b>")); display(fs_label)
display(HTML("<b>FHRPY false-signal detection</b>")); display(fs_pred)
link_scroll(fs_label, fs_pred)   # both viewers stay at the same instant

## 3. Expert labels vs prediction — **baseline / accel / decel (Ldb)**

Same idea on a **morphology** training record: the **expert** baseline +
acceleration (green) / deceleration (red) zones vs the **FHRPY WMFB** baseline +
detected accel/decel — scroll-synchronised.

In [ ]:
NAME = "morpho_train21"   # a morphology training record with expert labels
bl_label = ds.load_and_process(NAME, source="expert", height=480)
# Ldb comparison = baseline + accel/decel ONLY: suppress the contraction zones
# and the per-deceleration type markers in the prediction (match the expert side).
bl_pred  = ds.load_and_process(NAME, source="method", height=480,
                           analyze_contractions=False, analyze_decel_types=False)

display(HTML("<b>Expert baseline + accel/decel</b>")); display(bl_label)
display(HTML("<b>FHRPY WMFB baseline + accel/decel</b>")); display(bl_pred)
link_scroll(bl_label, bl_pred)

## 4. Going further — head-less methods, dynamic control, export

**(a) Run the methods without any UI**, inspect the numbers, then use them.

In [ ]:
from fhrpy.io import read_fhr
from fhrpy.baseline import analyze, compute_features
from fhrpy.viewer import FHRViewer

NAME = "ctg_example_01"
rec = read_fhr(ds.example_path(NAME))
# FHRMA pipeline ORDER: false signals are detected & REMOVED first, THEN the WMFB
# baseline is computed (on the cleaned signal — the scalp carries it where the
# Doppler is false). Pass false_signals="doppler" so analyze() does it for you.
ma = analyze(rec, false_signals="doppler")
print("baseline pts:", len(ma["baseline"]),
      "| acc:", len(ma["accelerations"]), "| dec:", len(ma["decelerations"]),
      "| contractions:", len(ma["contractions"]),
      "| false-signal episodes:", len(ma["false_signals"]["segments"]))

**(b) Feature synthesis.** `compute_features` returns the standard CTG
features (English names) — baseline / time-in-band, accel & decel morphology,
deceleration types, contractions and short-/long-term variability.

In [ ]:
feats = compute_features(ma)        # pass the analysis dict (no recompute)
print(len(feats), "features (all of them):")
for k, val in feats.items():
    print(f"  {k:36s} {round(val, 3) if isinstance(val, float) else val}")

**(c) Drive the viewer dynamically from Python.** The recording is already
read (`rec`) and analysed (`ma`), so build the viewer straight from them with
`FHRViewer.from_analysis` — **no reprocessing** — then steer it live.

In [ ]:
v = FHRViewer.from_analysis(rec, ma, channels=["FHR1", "MHR"], height=480)
display(v)

v.set_scale(3)                  # 3 cm/min paper speed
v.set_safe_zone(110, 150)       # move the central "safe" band
v.set_range(50, 210)            # top-grid bounds
v.scroll_to(0.5)                # jump to the middle
v.on("scroll", lambda d: print("scrolled to", round(d.get("time", 0) / 60, 1), "min"))
# also: set_contractions_visible, set_zones_visible, set_false_signals_visible,
#       set_marks_visible, set_interpolate, set_channel_visible, set_markers, print()

**(d) Export a standalone offline HTML** (no server, no PHP), reusing the
same analysis. Files go to a gitignored `generated/` folder, named after the
recording. The toolbar printer button gives a multi-page **A4-landscape PDF**
(~1 cm/min, 20 bpm/cm).

In [ ]:
import pathlib
out = pathlib.Path("generated"); out.mkdir(exist_ok=True)
path = out / f"{NAME}.html"
FHRViewer.from_analysis(rec, ma, channels=["FHR1", "MHR"], height=480,
                        rcf_min=50, rcf_max=210, safe_min=110, safe_max=160).to_html(path)
print("wrote", path)